**Cell 1**

# 04 — YOLOv11 Training

Picks a YOLOv11 variant and trains it on the unified dataset produced by notebook 02 (and health-checked in notebook 03).

| Variant   | Params | Speed | When to pick |
|-----------|-------:|------:|--------------|
| `yolo11n` | ~2.6 M | fastest | edge / demo |
| `yolo11s` | ~9.4 M | fast | **default for this project** |
| `yolo11m` | ~20 M | moderate | if accuracy plateaus |

We default to **`yolo11s`** — good balance on an 800-image dataset.

In [ ]:
# Cell 2 — install dependencies for this notebook
# Ultralytics pulls in torch, torchvision, opencv-python, numpy, pandas, matplotlib.
%pip install -q "ultralytics==8.3.*" pandas matplotlib
# On a bare Linux box (no system libGL) OpenCV import can fail — if so, uncomment:
# !apt-get update -qq && apt-get install -y -qq libgl1

In [ ]:
# Cell 3 — import
from ultralytics import YOLO
from pathlib import Path
import torch, shutil

DATA_YAML = Path('../data/dataset/data.yaml').resolve()
assert DATA_YAML.exists(), f'missing {DATA_YAML} — run notebook 02 first to build the unified dataset'
print('CUDA available:', torch.cuda.is_available())

**Cell 4**

## 4.1 Hyperparameters

In [ ]:
# Cell 5 — training config
CFG = dict(
    model='yolo11s.pt',     # try 'yolo11n.pt' or 'yolo11m.pt' to compare
    data=str(DATA_YAML),
    epochs=100,
    imgsz=640,
    batch=16,
    optimizer='SGD',
    lr0=0.01,
    patience=20,            # early stopping
    project='../runs/detect',
    name='campus_yolo11s',
    seed=42,
)
CFG

**Cell 6**

## 4.2 Train

In [ ]:
# Cell 7 — train YOLOv11s
model = YOLO(CFG['model'])
results = model.train(**{k: v for k, v in CFG.items() if k != 'model'})
print('run dir:', results.save_dir)

**Cell 8**

## 4.3 Training curves
Ultralytics writes `results.csv`; plot loss and mAP per epoch.

In [ ]:
# Cell 9 — plot training curves
import pandas as pd, matplotlib.pyplot as plt
run_dir = Path(results.save_dir)
df = pd.read_csv(run_dir / 'results.csv')
df.columns = [c.strip() for c in df.columns]

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
df.plot(x='epoch', y=['train/box_loss', 'train/cls_loss', 'val/box_loss', 'val/cls_loss'], ax=ax[0])
ax[0].set_title('Loss curves')
df.plot(x='epoch', y=['metrics/mAP50(B)', 'metrics/mAP50-95(B)'], ax=ax[1])
ax[1].set_title('Validation mAP')
plt.tight_layout(); plt.show()

**Cell 10**

## 4.4 Promote best checkpoint

In [ ]:
# Cell 11 — copy best.pt to weights/
weights_dir = Path('../weights'); weights_dir.mkdir(exist_ok=True)
best = run_dir / 'weights' / 'best.pt'
dst = weights_dir / 'best.pt'
shutil.copy2(best, dst)
print('saved', dst)

**Cell 12**

## 4.5 Quick sanity inference

In [ ]:
# Cell 13 — predict on a few test images
test_imgs = list(Path('../data/dataset/images/test').glob('*.jpg'))[:4]
m = YOLO(dst)
res = m.predict(source=[str(p) for p in test_imgs], imgsz=640, conf=0.25, save=True, project=str(run_dir), name='sanity')
print('predictions saved under', run_dir / 'sanity')

**Cell 14**

### Expected training indicators
- `val/box_loss` decreases smoothly and plateaus
- `mAP@0.5` > 0.80 on val for all 4 classes within ~60–80 epochs
- No runaway gap between train and val loss (overfitting sign)

Proceed to **notebook 05** for full evaluation.